Explanatory text

## Initialization

In [ ]:
# Imports

import pickle
from pathlib import Path
from typing import Any, Callable, Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
# from data_processing.processing.calibration import Detector, recalibrate
# from data_processing.processing.figure_of_merit import gaussian
# from data_processing.processing.neutron_classification import classify
# from data_processing.processing.neutron_window_generation import (
#     generate_nasa_neutron_window,
#     generate_n_distro_neutron_window
# )
from data_processing import processing as proc
from data_processing import loading as load
from data_processing import types as proc_types
# from data_processing.arc_paths import (INPUT_DATA_FOLDER, get_exp_root,
#                                        get_parq_root)
from data_processing.dataframe_validation import DetectorDataframeColumn, EnergyColumn
from data_processing.experiment_data_keys import (ExperimentDataKey,
                                                  ExperimentNeutronData)
from data_processing.helpers import (get_input_with_default,
                                     input_experiment_ids, stop)
# from data_processing.loading import get_neutron_window_paths, load_side_borders
# from data_processing.loading.dataframe_loading import load_psd
# from data_processing.loading.timetag_processing import calculate_timetag_hours
# from data_processing.reporting import plot_classification
# from data_processing.types import (BimodalBounds, BimodalParams,
#                                    NasaGenerationSettings,
#                                    NeutronWindowSettings, WindowType)
# from scipy.optimize import curve_fit
# from scipy.signal import deconvolve

In [ ]:
CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]

### Functions

In [ ]:
def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> proc_types.NasaGenerationSettings:
    sigma = get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee) (default)
2: newer (~0.1866 MeVee)
or press Enter for default
""",
            1,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = ExperimentDataKey.NASA_BORDERS if existing_left_border_version_input == 1 else ExperimentDataKey.NASA_BORDERS_RECALC
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = load.get_neutron_window_paths(file_name_prefix=file_name_prefix)
            left_border, _ = load.load_side_borders(side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.1966)
""",
            0.1966,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: proc.NeutronStrategyFactory,
    window_type: proc_types.WindowType,
    loading: bool,
    settings: proc_types.NeutronWindowSettings
) -> Callable[[], proc.AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], proc.AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data

In [ ]:
def get_light_output_converter() -> interp1d:
    interp_fn_path = Path() / "light_output_n_energy_rough_fn.pkl"
    l_ep_data_path = Path() / "l_ep_data.txt"

    try:
        with open(interp_fn_path, "rb") as fn_file:
            interpolator = pickle.load(fn_file)
    except (OSError, pickle.PickleError):
        result = np.loadtxt(l_ep_data_path)
        L, Ep = result.T
        interpolator = interp1d(L / 1000, Ep, kind="cubic", bounds_error=False)
        with open(interp_fn_path, "wb") as fn_file:
            pickle.dump(interpolator, fn_file)

    return interpolator

In [ ]:
def moving_average(arr, n=5):
    ret = np.cumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((n-1,))
    prefix[:] = np.nan
    return np.concatenate((prefix, mov_avg))

def moving_average_centered(arr, n=5):
    if n % 2 != 1:
        raise ValueError("Centered moving average needs odd window size")
    prefix_count = (n-1)//2
    ret = np.nancumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((prefix_count,))
    suffix = np.empty((prefix_count,))
    prefix[:] = np.nan
    suffix[:] = np.nan
    return np.concatenate((prefix, mov_avg, suffix))

In [ ]:
def separate_bin_edges(bin_edges):
    lo = bin_edges[:-1]
    hi = bin_edges[1:]
    return lo, hi


def get_bin_widths(bin_edges):
    lo, hi = separate_bin_edges(bin_edges)
    return hi-lo


def get_bin_mids(bin_edges):
    lo, hi = separate_bin_edges(bin_edges)
    return (hi+lo)/2

## Experiment ID Input

In [ ]:
experiment_ids = input_experiment_ids()

In [ ]:
# more here?
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

In [ ]:
calib_input = get_input_with_default(
    "Do you want to use new calibration? [y/n, or press Enter for yes]",
    "y",
    str
)

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column: EnergyColumn = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = ExperimentDataKey.NEW_CALIBRATION if is_new_calibration else ExperimentDataKey.CAEN_CALIBRATION

In [ ]:
strategy_factory = proc.NeutronStrategyFactory()
settings = get_nasa_generation_settings(calib_key)
factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, settings)
experiment_neutron_data = make_strategy_for_experiments(
    experiment_neutron_data, factory_fn)

## Data Loading and Initial Processing

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    exp_data[ExperimentDataKey.UNCLASSIFIED] = load.load_psd(exp_id)

In [ ]:
# Express timetags in hours elapsed
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = load.calculate_timetag_hours(unclassified_df)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Recalibrate energy
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = proc.recalibrate(unclassified_df, proc.Detector.ZERO)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

## Neutron Classification

In [ ]:
# Generate histogram

start_scan_idx = 0
end_scan_idx = 420
energy_width = 20e-3

for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    Z, xe, ye = proc.get_psd_energy_histogram(
        psd_report,
        calibrated_energy_column,
        energy_width=energy_width
    )
    exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
    exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
    exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
    exp_data[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))

In [ ]:
# TODO get fit dataframe (not needed if loading, but do anyway to keep process consistent)
stop_here = False

for exp_id, exp_data in experiment_neutron_data.items():
    Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
    xe = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    end_scan_idx = exp_data[ExperimentDataKey.END_SCAN_IDX]

    # # Default
    # default_bounds: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.25, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.38, 0.04, 4000)
    # )

    # bounds_a: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.04, 4000)
    # )

    # bounds_b: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.03, 4000)
    # )

    # # Ranged Example
    # bounds = [
    #     ((0, 60), bounds_a),
    # ]

    df, df_err = proc.scan_histogram_slices(
        Z,
        xe,
        ye,
        fit_style="peak_finder",
        # default_bounds,
        # bounds=bounds,
        start_idx=start_scan_idx,
        end_idx=end_scan_idx
    )
    df, bad_slice_indexes = proc.find_failed_slices(df, exp_id)

    if bad_slice_indexes is not None:
        exp_data[ExperimentDataKey.VALID_SLICE_FITS] = df
        exp_data[ExperimentDataKey.BAD_SLICE_INDEXES] = bad_slice_indexes
        stop_here = True
    else:
        # exp_data['fom_results'] = df
        exp_data[ExperimentDataKey.FOM_RESULTS] = df

if stop_here:
    stop()

In [ ]:
# get borders from strategy
for exp_id, exp_data in experiment_neutron_data.items():
    if ExperimentDataKey.FOM_RESULTS not in exp_data:
        print(f"No good fit data on Experiment {exp_id}")
        continue

    fom_results = exp_data[ExperimentDataKey.FOM_RESULTS]
    strategy = exp_data[ExperimentDataKey.BORDER_STRATEGY]

    strategy.set_slice_fit_dataframe(fom_results)
    borders = strategy.get_neutron_window()

    exp_data[ExperimentDataKey.BORDERS] = borders

In [ ]:
# classify neutrons
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED].copy()
    borders = exp_data[ExperimentDataKey.BORDERS]

    psd_report = proc.classify(
        psd_report,
        calibrated_energy_column,
        borders,
        DetectorDataframeColumn.NEW_N_CLASS
    )

    exp_data[ExperimentDataKey.PSD_REPORT] = psd_report

In [ ]:
# experiment_neutron_data["ID-419"].keys()

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    n_class_col_name = DetectorDataframeColumn.NEW_N_CLASS.value
    
    gamma_only = psd_report.query(f"~{n_class_col_name}").copy()
    neutrons_only = psd_report.query(n_class_col_name).copy()
    exp_data[ExperimentDataKey.NEUTRONS_ONLY] = neutrons_only
    exp_data[ExperimentDataKey.GAMMA_ONLY] = gamma_only

## Pulse Height Distribution

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    energy_bins = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    neutrons_only = exp_data[ExperimentDataKey.NEUTRONS_ONLY]
    print(neutrons_only.head())
    neutron_energies = neutrons_only[calibrated_energy_column.value]
    
    # TODO generate neutron PHD histogram
    Z, *_ = np.histogram(neutron_energies, bins=energy_bins)
    exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION] = {"standard": Z}

In [ ]:
# moving average
for exp_id, exp_data in experiment_neutron_data.items():
    phd_histogram_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]
    phd_histogram = phd_histogram_data["standard"]

    phd_moving_average = moving_average_centered(phd_histogram)

    phd_histogram_data["moving_average"] = phd_moving_average
    exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION] = phd_histogram_data

## Light Output to Neutron Energy

In [ ]:
# Get converter
converter = get_light_output_converter()

In [ ]:
# Convert light output to energy
for exp_id, exp_data in experiment_neutron_data.items():
    l_bins = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]

    energy_bins = converter(l_bins)

    exp_data[ExperimentDataKey.HISTOGRAM_ENERGY_BIN_EDGES] = energy_bins

## Display and Output

### Display

In [ ]:
# Light output
for exp_id, exp_data in experiment_neutron_data.items():
    l_bins = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    phd_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]
    phd_histogram = phd_data["standard"]
    phd_mov_avg = phd_data["moving_average"]

    bins_lo_edge, _ = separate_bin_edges(l_bins)
    bin_widths = get_bin_widths(l_bins)

    fig, ax = plt.subplots(figsize=(8, 8))

    ax.bar(
        bins_lo_edge,
        phd_histogram,
        width=bin_widths,
        align="edge",
        alpha=0.2,
        label="Raw"
    )
    ax.bar(
        bins_lo_edge,
        phd_mov_avg,
        width=bin_widths,
        align="edge",
        alpha=0.2,
        label="Moving Average"
    )

    ax.set_xlim(0, 1.5)

    ax.set_title(f"{exp_id} Pulse Height Distribution")
    ax.set_xlabel("Light output (MeVee)")
    ax.set_ylabel("Neutron count")
    ax.legend()

    plt.show()

In [ ]:
# Neutron energy
for exp_id, exp_data in experiment_neutron_data.items():
    energy_bins = exp_data[ExperimentDataKey.HISTOGRAM_ENERGY_BIN_EDGES]
    phd_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]
    phd_histogram = phd_data["standard"]
    phd_mov_avg = phd_data["moving_average"]

    bins_lo_edge, _ = separate_bin_edges(energy_bins)
    bin_widths = get_bin_widths(energy_bins)

    fig, ax = plt.subplots(figsize=(8, 8))

    ax.bar(
        bins_lo_edge,
        phd_histogram,
        width=bin_widths,
        align="edge",
        alpha=0.2,
        label="Raw"
    )
    ax.bar(
        bins_lo_edge,
        phd_mov_avg,
        width=bin_widths,
        align="edge",
        alpha=0.2,
        label="Moving Average"
    )

    ax.set_title(f"{exp_id} Pulse Height Distribution")
    ax.set_xlabel("Neutron energy (MeV)")
    ax.set_ylabel("Neutron count")
    ax.legend()

    plt.show()

### Output

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    # TODO get all data
    l_bins = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    energy_bins = exp_data[ExperimentDataKey.HISTOGRAM_ENERGY_BIN_EDGES]
    phd_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]
    phd_histogram = phd_data["standard"]
    phd_mov_avg = phd_data["moving_average"]

    # TODO create dataframe
    l_lo_edges, l_hi_edges = separate_bin_edges(l_bins)
    en_lo_edges, en_hi_edges = separate_bin_edges(energy_bins)

    data = {
        "Light output bin start (MeVee)": l_lo_edges,
        "Light output bin end (MeVee)": l_hi_edges,
        "Energy bin start (MeV)": en_lo_edges,
        "Energy bin end (MeV)": en_hi_edges,
        "Neutron counts": phd_histogram,
        # "Neutron moving average": phd_mov_avg
    }
    export_df = pd.DataFrame(data=data)

    # TODO save to CSV
    phd_input_path = Path() / "pulse_height_distribution" / "input"
    csv_path = phd_input_path / f"{exp_id}-n_spectrum.csv"
    export_df.to_csv(csv_path, index_label="Index")
    print(f"Saved output data at {csv_path}")

In [ ]:
input("Processing done, hit Enter to finish")
stop()